In [1]:
%pip install transformers torch

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import torch
from transformers import pipeline

print("GPU disponível:", torch.cuda.is_available())

c:\Users\Carol\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU disponível: False


In [3]:
# A verificação retornou False para CUDA; o teste utilizará a CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    dtype=torch.float32,
    device=device,
)

print("Modelo carregado em:", device)

Loading weights: 100%|██████████| 290/290 [00:03<00:00, 88.36it/s] 


Modelo carregado em: cpu


In [4]:
#teste 1
import time
from transformers import GenerationConfig

prompt = pipe.tokenizer.apply_chat_template(
    [
        {
            "role": "system",
            "content": "Você explica dados dos Juizados Especiais em português."
        },
        {
            "role": "user",
            "content": """Escreva uma frase comparando os tempos médios
até a sentença. Use somente os dados fornecidos.

Ano: 2025
TJDFT: 90,1 dias
TJSE: 111,7 dias
Diferença já calculada: 21,6 dias."""
        }
    ],
    tokenize=False,
    add_generation_prompt=True,
)

inicio = time.time()

resposta = pipe(
    prompt,
    generation_config=GenerationConfig(
        max_new_tokens=100,
        do_sample=False,
    ),
    return_full_text=False,
)

print(resposta[0]["generated_text"])
print("Tempo:", round(time.time() - inicio, 1), "segundos")

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Em 2025, o tempo médio para julgamento de acórdãos do Tribunal Superior Eleitoral (TSE) foi de 111,7 dias, enquanto o do Juizado Especial da Justiça Federal (JEF) era de 90,1 dias. A diferença entre esses dois períodos é de 21,6 dias.
Tempo: 11.8 segundos


In [5]:
#teste 2 com refinamento do prompt
prompt = pipe.tokenizer.apply_chat_template(
    [
        {
            "role": "system",
            "content": "Você explica dados dos Juizados Especiais em português."
        },
        {
            "role": "user",
            "content": """CONTEXTO
Os dados comparam o tempo médio até a sentença nos Juizados Especiais.
As siglas identificam os tribunais:
TJDFT: Distrito Federal.
TJSE: Sergipe.

TAREFA
Compare os tempos e informe qual tribunal apresentou o menor
tempo médio até a sentença e qual foi a diferença.
Mantenha exatamente as siglas TJDFT e TJSE, sem expandi-las.
Utilize somente as informações fornecidas.

DADOS
Ano: 2025
TJDFT: 90,1 dias
TJSE: 111,7 dias
Diferença já calculada: 21,6 dias

FORMATO
Uma única frase em português."""
        }
    ],
    tokenize=False,
    add_generation_prompt=True,
)

inicio = time.time()

resposta = pipe(
    prompt,
    generation_config=GenerationConfig(
        max_new_tokens=100,
        do_sample=False,
    ),
    return_full_text=False,
)

print(resposta[0]["generated_text"])
print("Tempo:", round(time.time() - inicio, 1), "segundos")

A tarefa de comparar os tempos médios de sentença entre os Juizados Especiais no Distrito Federal (TJDFT) e no Estado do Sergipe (TJSE) é uma tarefa relevante para compreender a distribuição das penas aplicadas em diferentes regiões da justiça federal brasileira. A diferença de 21,6 dias entre os dois tribunais pode ser vista como um indicativo da complexidade e variação
Tempo: 11.9 segundos


In [6]:
#teste 3 com o mesmo refinamento do prompt anterior, testando um novo modelo

pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    dtype=torch.bfloat16,
    device=torch.device("cpu"),
)

print("Modelo 1.5B carregado na CPU.")

prompt = pipe.tokenizer.apply_chat_template(
    [
        {
            "role": "system",
            "content": "Você explica dados dos Juizados Especiais em português."
        },
        {
            "role": "user",
            "content": """CONTEXTO
Os dados comparam o tempo médio até a sentença nos Juizados Especiais.
As siglas identificam os tribunais:
TJDFT: Distrito Federal.
TJSE: Sergipe.

TAREFA
Compare os tempos e informe qual tribunal apresentou o menor
tempo médio até a sentença e qual foi a diferença.
Mantenha exatamente as siglas TJDFT e TJSE, sem expandi-las.
Utilize somente as informações fornecidas.

DADOS
Ano: 2025
TJDFT: 90,1 dias
TJSE: 111,7 dias
Diferença já calculada: 21,6 dias

FORMATO
Uma única frase em português."""
        }
    ],
    tokenize=False,
    add_generation_prompt=True,
)

inicio = time.time()

resposta = pipe(
    prompt,
    generation_config=GenerationConfig(
        max_new_tokens=100,
        do_sample=False,
    ),
    return_full_text=False,
)

print(resposta[0]["generated_text"])
print("Tempo:", round(time.time() - inicio, 1), "segundos")

Loading weights: 100%|██████████| 338/338 [00:06<00:00, 50.05it/s]


Modelo 1.5B carregado na CPU.
Em 2025, o Tribunal de Justiça do Estado de Sergipe (TJSE) apresentou o menor tempo médio até a sentença, sendo 111,7 dias contra 90,1 dias no Tribunal Regional do Direito do Distrito Federal e de Territórios (TJDFT). A diferença entre esses tempos é de 21,6 dias.
Tempo: 17.5 segundos


In [7]:
#teste 4 com novo prompt
prompt = pipe.tokenizer.apply_chat_template(
    [
        {
            "role": "system",
            "content": "Você explica dados dos Juizados Especiais em português."
        },
        {
            "role": "user",
            "content": """TAREFA
Transforme o resultado abaixo em uma única frase em português.
Preserve os valores e a indicação de menor e maior tempo.
Use apenas as siglas fornecidas para identificar os tribunais.

RESULTADO DA COMPARAÇÃO
Ano: 2025
Indicador: tempo médio até a sentença nos Juizados Especiais
Tribunal com menor tempo: TJDFT
Tempo do TJDFT: 90,1 dias
Tribunal com maior tempo: TJSE
Tempo do TJSE: 111,7 dias
Diferença: 21,6 dias

FORMATO
Somente a frase solicitada, sem expandir as siglas nem
acrescentar informações."""
        }
    ],
    tokenize=False,
    add_generation_prompt=True,
)

inicio = time.time()

resposta = pipe(
    prompt,
    generation_config=GenerationConfig(
        max_new_tokens=100,
        do_sample=False,
    ),
    return_full_text=False,
)

print(resposta[0]["generated_text"])
print("Tempo:", round(time.time() - inicio, 1), "segundos")

Em 2025, o Tribunal de Justiça Distrital Federal (TJDFT) tem um tempo médio até a sentença de 90,1 dias, enquanto o Tribunal de Justiça do Estado de São Paulo (TJSE) leva 111,7 dias, resultando em uma diferença de 21,6 dias.
Tempo: 13.5 segundos


In [8]:
#teste 5 com exemplo de saída
prompt = pipe.tokenizer.apply_chat_template(
    [
        {
            "role": "system",
            "content": "Você explica dados dos Juizados Especiais em português."
        },
        {
            "role": "user",
            "content": """TAREFA
Transforme o resultado abaixo em uma única frase em português.
Preserve os valores e a indicação de menor e maior tempo.
Use apenas as siglas fornecidas para identificar os tribunais.

EXEMPLO FICTÍCIO DE FORMATO
Dados: ano 2024; menor tempo: TJA, 100 dias;
maior tempo: TJB, 150 dias; diferença: 50 dias.

Resposta: Em 2024, o TJA apresentou tempo médio até a sentença
de 100 dias, ficando 50 dias abaixo do TJB, que registrou 150 dias.

O exemplo mostra somente o formato.
Escreva sua resposta usando os dados da seção
RESULTADO DA COMPARAÇÃO abaixo.

RESULTADO DA COMPARAÇÃO
Ano: 2025
Indicador: tempo médio até a sentença nos Juizados Especiais
Tribunal com menor tempo: TJDFT
Tempo do TJDFT: 90,1 dias
Tribunal com maior tempo: TJSE
Tempo do TJSE: 111,7 dias
Diferença: 21,6 dias

FORMATO
Somente a frase solicitada, sem expandir as siglas nem
acrescentar informações."""
        }
    ],
    tokenize=False,
    add_generation_prompt=True,
)

inicio = time.time()

resposta = pipe(
    prompt,
    generation_config=GenerationConfig(
        max_new_tokens=100,
        do_sample=False,
    ),
    return_full_text=False,
)

print(resposta[0]["generated_text"])
print("Tempo:", round(time.time() - inicio, 1), "segundos")

Em 2025, o Tribunal de Justiça Distrital Federal (TJDFT) apresentou um tempo médio até a sentença de 90,1 dias, ficando 21,6 dias acima do Tribunal de Justiça Estadual (TJSE), que registrou 111,7 dias.
Tempo: 11.8 segundos


In [9]:
#teste 6 com um novo modelo repetindo o teste anterior com exemplo de saída
import time
import torch
from transformers import pipeline, GenerationConfig

pipe = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    dtype=torch.bfloat16,
    device=torch.device("cpu"),
)

print("TinyLlama carregado na CPU.")

prompt = pipe.tokenizer.apply_chat_template(
    [
        {
            "role": "system",
            "content": "Você explica dados dos Juizados Especiais em português."
        },
        {
            "role": "user",
            "content": """TAREFA
Transforme o resultado abaixo em uma única frase em português.
Preserve os valores e a indicação de menor e maior tempo.
Use apenas as siglas fornecidas para identificar os tribunais.

EXEMPLO FICTÍCIO DE FORMATO
Dados: ano 2024; menor tempo: TJA, 100 dias;
maior tempo: TJB, 150 dias; diferença: 50 dias.

Resposta: Em 2024, o TJA apresentou tempo médio até a sentença
de 100 dias, ficando 50 dias abaixo do TJB, que registrou 150 dias.

O exemplo mostra somente o formato.
Escreva sua resposta usando os dados da seção
RESULTADO DA COMPARAÇÃO abaixo.

RESULTADO DA COMPARAÇÃO
Ano: 2025
Indicador: tempo médio até a sentença nos Juizados Especiais
Tribunal com menor tempo: TJDFT
Tempo do TJDFT: 90,1 dias
Tribunal com maior tempo: TJSE
Tempo do TJSE: 111,7 dias
Diferença: 21,6 dias

FORMATO
Somente a frase solicitada, sem expandir as siglas nem
acrescentar informações."""
        }
    ],
    tokenize=False,
    add_generation_prompt=True,
)

inicio = time.time()

resposta = pipe(
    prompt,
    generation_config=GenerationConfig(
        max_new_tokens=100,
        do_sample=False,
    ),
    return_full_text=False,
)

print(resposta[0]["generated_text"])
print("Tempo:", round(time.time() - inicio, 1), "segundos")

Loading weights: 100%|██████████| 201/201 [00:02<00:00, 91.03it/s]
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TinyLlama carregado na CPU.
Dados: ano 2025; menor tempo: TJDFT, 90,1 dias; maior tempo: TJSE, 111,7 dias; diferença: 21,6 dias.
Resposta: Em 2025, o TJDFT apresentou tempo médio até a sentença de 90,1 dias, ficando 21,6 dias abaixo
Tempo: 12.3 segundos


In [10]:
#Reexecução do teste 6 aumentando para 256 o limite de tokens
prompt = pipe.tokenizer.apply_chat_template(
    [
        {
            "role": "system",
            "content": "Você explica dados dos Juizados Especiais em português."
        },
        {
            "role": "user",
            "content": """TAREFA
Transforme o resultado abaixo em uma única frase em português.
Preserve os valores e a indicação de menor e maior tempo.
Use apenas as siglas fornecidas para identificar os tribunais.

EXEMPLO FICTÍCIO DE FORMATO
Dados: ano 2024; menor tempo: TJA, 100 dias;
maior tempo: TJB, 150 dias; diferença: 50 dias.

Resposta: Em 2024, o TJA apresentou tempo médio até a sentença
de 100 dias, ficando 50 dias abaixo do TJB, que registrou 150 dias.

O exemplo mostra somente o formato.
Escreva sua resposta usando os dados da seção
RESULTADO DA COMPARAÇÃO abaixo.

RESULTADO DA COMPARAÇÃO
Ano: 2025
Indicador: tempo médio até a sentença nos Juizados Especiais
Tribunal com menor tempo: TJDFT
Tempo do TJDFT: 90,1 dias
Tribunal com maior tempo: TJSE
Tempo do TJSE: 111,7 dias
Diferença: 21,6 dias

FORMATO
Somente a frase solicitada, sem expandir as siglas nem
acrescentar informações."""
        }
    ],
    tokenize=False,
    add_generation_prompt=True,
)

inicio = time.time()

resposta = pipe(
    prompt,
    generation_config=GenerationConfig(
        max_new_tokens=256,
        do_sample=False,
    ),
    return_full_text=False,
)

print(resposta[0]["generated_text"])
print("Tempo:", round(time.time() - inicio, 1), "segundos")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Dados: ano 2025; menor tempo: TJDFT, 90,1 dias; maior tempo: TJSE, 111,7 dias; diferença: 21,6 dias.
Resposta: Em 2025, o TJDFT apresentou tempo médio até a sentença de 90,1 dias, ficando 21,6 dias abaixo do TJSE, que registrou 111,7 dias.
Tempo: 13.4 segundos


In [11]:
import time
import torch
from transformers import pipeline, GenerationConfig

pipe = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    dtype=torch.bfloat16,
    device=torch.device("cpu"),
)

#Reexecução do teste 6 
prompt = pipe.tokenizer.apply_chat_template(
    [
        {
            "role": "system",
            "content": "Você explica dados dos Juizados Especiais em português."
        },
        {
            "role": "user",
            "content": """TAREFA
Transforme o resultado abaixo em uma única frase em português.
Preserve os valores e a indicação de menor e maior tempo.
Use apenas as siglas fornecidas para identificar os tribunais.

EXEMPLO FICTÍCIO DE FORMATO
Dados: ano 2024; menor tempo: TJA, 100 dias;
maior tempo: TJB, 150 dias; diferença: 50 dias.

Resposta: Em 2024, o TJA apresentou tempo médio até a sentença
de 100 dias, ficando 50 dias abaixo do TJB, que registrou 150 dias.

O exemplo mostra somente o formato.
Escreva sua resposta usando os dados da seção
RESULTADO DA COMPARAÇÃO abaixo.

RESULTADO DA COMPARAÇÃO
Ano: 2025
Indicador: tempo médio até a sentença nos Juizados Especiais
Tribunal com menor tempo: TJDFT
Tempo do TJDFT: 90,1 dias
Tribunal com maior tempo: TJSE
Tempo do TJSE: 111,7 dias
Diferença: 21,6 dias

FORMATO
Somente a frase solicitada, sem expandir as siglas nem
acrescentar informações."""
        }
    ],
    tokenize=False,
    add_generation_prompt=True,
)

inicio = time.time()

resposta = pipe(
    prompt,
    generation_config=GenerationConfig(
        max_new_tokens=256,
        do_sample=False,
    ),
    return_full_text=False,
)

print(resposta[0]["generated_text"])
print("Tempo:", round(time.time() - inicio, 1), "segundos")

Loading weights: 100%|██████████| 201/201 [00:02<00:00, 80.58it/s]
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Dados: ano 2025; menor tempo: TJDFT, 90,1 dias; maior tempo: TJSE, 111,7 dias; diferença: 21,6 dias.
Resposta: Em 2025, o TJDFT apresentou tempo médio até a sentença de 90,1 dias, ficando 21,6 dias abaixo do TJSE, que registrou 111,7 dias.
Tempo: 13.9 segundos


In [12]:
#Reexecução do teste 6 com outro tribunal
prompt = pipe.tokenizer.apply_chat_template(
    [
        {
            "role": "system",
            "content": "Você explica dados dos Juizados Especiais em português."
        },
        {
            "role": "user",
            "content": """TAREFA
Transforme o resultado abaixo em uma única frase em português.
Preserve os valores e a indicação de menor e maior tempo.
Use apenas as siglas fornecidas para identificar os tribunais.

EXEMPLO FICTÍCIO DE FORMATO
Dados: ano 2024; menor tempo: TJA, 100 dias;
maior tempo: TJB, 150 dias; diferença: 50 dias.

Resposta: Em 2024, o TJA apresentou tempo médio até a sentença
de 100 dias, ficando 50 dias abaixo do TJB, que registrou 150 dias.

O exemplo mostra somente o formato.
Escreva sua resposta usando os dados da seção
RESULTADO DA COMPARAÇÃO abaixo.

RESULTADO DA COMPARAÇÃO
Ano: 2025
Indicador: tempo médio até a sentença nos Juizados Especiais
Tribunal com menor tempo: TJBA
Tempo do TJBA: 120,0 dias
Tribunal com maior tempo: TJMA
Tempo do TJMA: 125,7 dias
Diferença: 5,7 dias

FORMATO
Somente a frase solicitada, sem expandir as siglas nem
acrescentar informações."""
        }
    ],
    tokenize=False,
    add_generation_prompt=True,
)

inicio = time.time()

resposta = pipe(
    prompt,
    generation_config=GenerationConfig(
        max_new_tokens=256,
        do_sample=False,
    ),
    return_full_text=False,
)

print(resposta[0]["generated_text"])
print("Tempo:", round(time.time() - inicio, 1), "segundos")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Dados: ano 2025; menor tempo: TJBA, 120,0 dias; maior tempo: TJMA, 125,7 dias; diferença: 5,7 dias.
Resposta: Em 2025, o TJBA apresentou tempo médio até a sentença de 120 dias, ficando 5,7 dias abaixo do TJMA, que registrou 125,7 dias.
Tempo: 14.2 segundos


In [13]:
print(
    "Memória dos parâmetros:",
    round(
        sum(p.numel() * p.element_size() for p in pipe.model.parameters())
        / 1000000000,
        2,
    ),
    "GB",
)

Memória dos parâmetros: 2.2 GB


In [14]:
# Teste de geração com a conexão de internet desativada.
inicio = time.time()

resposta = pipe(
    prompt,
    generation_config=GenerationConfig(
        max_new_tokens=256,
        do_sample=False,
    ),
    return_full_text=False,
)

print(resposta[0]["generated_text"])
print("Tempo:", round(time.time() - inicio, 1), "segundos")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Dados: ano 2025; menor tempo: TJBA, 120,0 dias; maior tempo: TJMA, 125,7 dias; diferença: 5,7 dias.
Resposta: Em 2025, o TJBA apresentou tempo médio até a sentença de 120 dias, ficando 5,7 dias abaixo do TJMA, que registrou 125,7 dias.
Tempo: 12.7 segundos


# TP5 - IMPLEMENTAÇÃO DA MEMORIA CONVERSACIONAL

In [2]:
import torch
from transformers import GenerationConfig, pipeline

pipe = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    dtype=torch.bfloat16,
    device=torch.device("cpu"),
)

pipe.model.generation_config.max_length = None

print("TinyLlama carregado na CPU.")

c:\Users\Carol\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:01<00:00, 124.09it/s]


TinyLlama carregado na CPU.


In [3]:
# Teste inicial: cada chamada recebe somente a pergunta atual.

sistema = """Você responde perguntas sobre os Juizados Especiais em português.
Use somente os fatos fornecidos, sem expandir as siglas.
Se faltar informação para entender a pergunta, peça esclarecimento.
Responda em uma frase curta.

TEMPO MÉDIO ATÉ A SENTENÇA EM 2025
TJDFT: 90,1 dias.
TJSE: 111,7 dias.
TJBA: 120,0 dias.
"""

for pergunta in [
    "Qual é o tempo médio até a sentença do TJSE em 2025?",
    "Qual tribunal eu citei na pergunta anterior?",
]:
    mensagens = [
        {"role": "system", "content": sistema},
        {"role": "user", "content": pergunta},
    ]

    prompt = pipe.tokenizer.apply_chat_template(
        mensagens,
        tokenize=False,
        add_generation_prompt=True,
    )

    print("\nPergunta:", pergunta)
    print("Quantidade de mensagens enviadas:", len(mensagens))
    print(
        "Resposta:",
        pipe(
            prompt,
            max_length=None,
            generation_config=GenerationConfig(
                max_new_tokens=256,
                do_sample=False,
            ),
            return_full_text=False,
        )[0]["generated_text"],
    )


Pergunta: Qual é o tempo médio até a sentença do TJSE em 2025?
Quantidade de mensagens enviadas: 2
Resposta: O tempo médio até a sentença do TJSE em 2025 é de 111,7 dias.

Pergunta: Qual tribunal eu citei na pergunta anterior?
Quantidade de mensagens enviadas: 2
Resposta: A pergunta anterior foi "Qual tribunal é responsável por julgar os juízes especiais?" A resposta para esta pergunta foi "O Tribunal de Justiça da República do Brasil é responsável por julgar os juízes especiais."


In [4]:
# Teste com histórico: preservar as perguntas e respostas anteriores.
historico = []

for pergunta in [
    "Qual é o tempo médio até a sentença do TJSE em 2025?",
    "Qual tribunal eu citei na pergunta anterior?",
]:
    mensagens = [{"role": "system", "content": sistema}] + historico + [
        {"role": "user", "content": pergunta},
    ]

    prompt = pipe.tokenizer.apply_chat_template(
        mensagens,
        tokenize=False,
        add_generation_prompt=True,
    )

    resposta = pipe(
        prompt,
        max_length=None,
        generation_config=GenerationConfig(
            max_new_tokens=256,
            do_sample=False,
        ),
        return_full_text=False,
    )[0]["generated_text"]

    print("\nPergunta:", pergunta)
    print("Quantidade de mensagens enviadas:", len(mensagens))
    print("Resposta:", resposta)

    historico.append({"role": "user", "content": pergunta})
    historico.append({"role": "assistant", "content": resposta})


Pergunta: Qual é o tempo médio até a sentença do TJSE em 2025?
Quantidade de mensagens enviadas: 2
Resposta: O tempo médio até a sentença do TJSE em 2025 é de 111,7 dias.

Pergunta: Qual tribunal eu citei na pergunta anterior?
Quantidade de mensagens enviadas: 4
Resposta: O tribunal que foi citado na pergunta anterior é o TJDFT.


In [5]:
# Refinamento: distinguir os fatos da base do histórico da conversa.
sistema_refinado = sistema + """
COMO LER A CONVERSA
Os fatos numéricos acima não representam perguntas feitas pelo usuário.
"Pergunta anterior" é a mensagem de role user imediatamente anterior à pergunta atual.
Se a pergunta pedir qual tribunal o usuário citou antes, consulte essa mensagem anterior.
Nesse caso, responda somente com a sigla que aparece nessa mensagem.
"""

historico = []

for pergunta in [
    "Qual é o tempo médio até a sentença do TJSE em 2025?",
    "Qual tribunal eu citei na pergunta anterior?",
]:
    mensagens = [
        {"role": "system", "content": sistema_refinado},
    ] + historico + [
        {"role": "user", "content": pergunta},
    ]

    prompt = pipe.tokenizer.apply_chat_template(
        mensagens,
        tokenize=False,
        add_generation_prompt=True,
    )

    resposta = pipe(
        prompt,
        max_length=None,
        generation_config=GenerationConfig(
            max_new_tokens=256,
            do_sample=False,
        ),
        return_full_text=False,
    )[0]["generated_text"]

    print("\nPergunta:", pergunta)
    print("Quantidade de mensagens enviadas:", len(mensagens))
    print("Resposta:", resposta)

    historico.append({"role": "user", "content": pergunta})
    historico.append({"role": "assistant", "content": resposta})


Pergunta: Qual é o tempo médio até a sentença do TJSE em 2025?
Quantidade de mensagens enviadas: 2
Resposta: O tempo médio até a sentença do TJSE em 2025 é de 111,7 dias.

Pergunta: Qual tribunal eu citei na pergunta anterior?
Quantidade de mensagens enviadas: 4
Resposta: A pergunta anterior que você citou foi "Como ler a conversa" e a mensagem de role usuário imediatamente anterior à pergunta atual é "Pergunta anterior". A mensagem anterior contém a sigla do tribunal que você citou na pergunta anterior.


In [6]:
# O histórico foi enviado, mas o modelo errou o tribunal anterior.
# Agora acrescentamos notas extraídas das perguntaS

historico = []
notas = []

for pergunta in [
    "Qual é o tempo médio até a sentença do TJSE em 2025?",
    "Qual tribunal eu citei na pergunta anterior?",
]:
    contexto = sistema

    if notas:
        contexto += (
            "\nTribunais mencionados nas perguntas anteriores: "
            + ", ".join(notas)
        )

    mensagens = [{"role": "system", "content": contexto}] + historico + [
        {"role": "user", "content": pergunta},
    ]

    prompt = pipe.tokenizer.apply_chat_template(
        mensagens,
        tokenize=False,
        add_generation_prompt=True,
    )

    resposta = pipe(
        prompt,
        max_length=None,
        generation_config=GenerationConfig(
            max_new_tokens=256,
            do_sample=False,
        ),
        return_full_text=False,
    )[0]["generated_text"]

    print("\nPergunta:", pergunta)
    print("Quantidade de mensagens enviadas:", len(mensagens))
    print("Notas enviadas:", notas)
    print("Resposta:", resposta)

    historico.append({"role": "user", "content": pergunta})
    historico.append({"role": "assistant", "content": resposta})

    for sigla in ["TJDFT", "TJSE", "TJBA"]:
        if sigla in pergunta:
            notas.append(sigla)


Pergunta: Qual é o tempo médio até a sentença do TJSE em 2025?
Quantidade de mensagens enviadas: 2
Notas enviadas: []
Resposta: O tempo médio até a sentença do TJSE em 2025 é de 111,7 dias.

Pergunta: Qual tribunal eu citei na pergunta anterior?
Quantidade de mensagens enviadas: 4
Notas enviadas: ['TJSE']
Resposta: O tribunal que foi citado na pergunta anterior é o TJSE.


In [7]:
# teste refeito com nova pergunta sobre outro tribunal

historico = []
notas = []

for pergunta in [
    "Qual é o tempo médio até a sentença do TJBA em 2025?",
    "Qual tribunal eu citei na pergunta anterior?",
]:
    contexto = sistema

    if notas:
        contexto += (
            "\nTribunais mencionados nas perguntas anteriores: "
            + ", ".join(notas)
        )

    mensagens = [{"role": "system", "content": contexto}] + historico + [
        {"role": "user", "content": pergunta},
    ]

    prompt = pipe.tokenizer.apply_chat_template(
        mensagens,
        tokenize=False,
        add_generation_prompt=True,
    )

    resposta = pipe(
        prompt,
        max_length=None,
        generation_config=GenerationConfig(
            max_new_tokens=256,
            do_sample=False,
        ),
        return_full_text=False,
    )[0]["generated_text"]

    print("\nPergunta:", pergunta)
    print("Quantidade de mensagens enviadas:", len(mensagens))
    print("Notas enviadas:", notas)
    print("Resposta:", resposta)

    historico.append({"role": "user", "content": pergunta})
    historico.append({"role": "assistant", "content": resposta})

    for sigla in ["TJDFT", "TJSE", "TJBA"]:
        if sigla in pergunta:
            notas.append(sigla)


Pergunta: Qual é o tempo médio até a sentença do TJBA em 2025?
Quantidade de mensagens enviadas: 2
Notas enviadas: []
Resposta: O tempo médio até a sentença do TJBA em 2025 é de 120,0 dias.

Pergunta: Qual tribunal eu citei na pergunta anterior?
Quantidade de mensagens enviadas: 4
Notas enviadas: ['TJBA']
Resposta: A pergunta anterior referiu-se a tribunais mencionados nas perguntas anteriores. A pergunta foi "Tribunais mencionados nas perguntas anteriores: TJBA, TJSE, e TJDFT". A resposta para a pergunta anterior foi "TJBA".


In [10]:
%pip install langchain_openai

  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached uuid_utils-1.2.0-cp310-abi3-win_amd64.whl.metadata (6.5 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached orjson-3.13.0-cp314-cp314-win_amd64.whl.metadata (43 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
  Using cached uuid_utils-0.19.0-cp310-abi3-win_amd64.whl.metadata (6.5 kB)
  Using cached xxhash-4.1.0-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached zstandard-0.25.0-cp314-cp314-win_amd64.whl.metadata (3.3 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached jiter-0.19.0-cp314-cp314-win_amd64.whl.metadata (5.3 kB)
   ---------------------------------------- 0.0/591.1 kB ? eta -:--:--
   ---------------------------------------- 591.1/591.1 kB 27.6 MB/s  0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 37.3 MB/s  0:00:00


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip
